# NB01 — Exploration, statistiques et PWM des séquences VHH
**Projet 1.1 — Phase A, tâches 1.3 à 1.6**

## Rôle de ce notebook
Le **nettoyage n'est plus fait ici** : il est fait par `scripts/prepare_vhh.py` (notebook `NB01b_prepare_vhh`, couches L1–L6 + déduplication exacte). NB01 lit le résultat et fait uniquement les **statistiques, la composition, la PWM et l'information par position**.

| | Chemin |
|---|---|
| Entrées | `data/processed/<DATA_VERSION>/vhh_clean_meta.csv`, `cleaning_report.csv`, `length_histogram_before_length_filter.csv`, `manifest.json` (produits par `prepare_vhh.py`) |
| Sortie | `data/processed/<DATA_VERSION>/nb01_stats.json` (le `manifest.json` n'est **pas** modifié) |
| Figures | `results/NB01/<DATA_VERSION>/` |

`DATA_VERSION` : `toy` en local, `v1` sur Colab par défaut. **Pour les nouvelles données, mettre `DATA_VERSION_OVERRIDE = "v2"`** (cellule 1), de même dans NB02.

**Avant NB01 en local (toy)** : `python scripts/make_toy_data.py` puis `python scripts/prepare_vhh.py --inputs data/toy/raw/toy_oas_raw.csv`.

## Formules (exactes, sans epsilon)
- Fréquence : f(a) = count(a) / N_total
- Entropie par position : H(i) = −Σₐ pᵢ(a) log₂ pᵢ(a), avec 0·log 0 = 0 (masque exact)
- PWM : PWM(i, a) = log₂( pᵢ(a) / p_bg(a) ) ; −∞ si pᵢ(a) = 0 (gris)
- Information : I(i) = log₂(20) − H(i) (Schneider & Stephens, 1990)
- Redondance exacte : (N − N_uniques) / N, N = lignes avant déduplication

## Références
Olsen et al. 2022 (OAS) · Schneider & Stephens 1990 (logos) · Fu et al. 2012 (CD-HIT)

In [ ]:
# ==============================================================================
# Cellule 1 : RÉGLAGES + bootstrap (local ET Google Colab), chemins, version des données
# ==============================================================================
# ------------------------------ RÉGLAGES (à modifier ici) ---------------------------
# Dossier data/processed/<nom>/ produit par prepare_vhh.py (ex. "v2").
#   None -> automatique : 'toy' en local, 'v1' sur Colab.
DATA_VERSION_OVERRIDE = None
# ------------------------------------------------------------------------------------
import os, sys, subprocess, re
from pathlib import Path

REPO_URL = "https://github.com/AhmadBelhaj4/Recherche_AI.git"
BRANCH = "main"


def _find_root(start):
    """Remonte depuis `start` jusqu'au dossier qui contient utils.py."""
    start = Path(start).resolve()
    for p in [start, *start.parents]:
        if (p / "utils.py").exists():
            return p
        if (p / "project1" / "utils.py").exists():
            return p / "project1"
    return None


try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
ROOT_DIR = _find_root(Path.cwd())

if ROOT_DIR is None and IN_COLAB:
    clone_target = Path("/content/Recherche_AI")
    if not clone_target.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(clone_target)], check=True)
    ROOT_DIR = _find_root(clone_target)

if ROOT_DIR is None:
    raise FileNotFoundError("utils.py introuvable : ouvrez le notebook dans le dépôt Recherche_AI.")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    _drive = Path("/content/drive/MyDrive/AI_Project_Shared")
    if not _drive.exists():
        raise FileNotFoundError(f"Dossier Drive introuvable : {_drive}. Ajoutez le raccourci dans Mon Drive.")
    os.environ.setdefault("AI_PROJECT_DIR", str(_drive))

sys.path.insert(0, str(ROOT_DIR))

import json, hashlib
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import get_paths, get_data_version, set_seed, STANDARD_AA, compute_aa_composition

SEED = 42
set_seed(SEED)

if DATA_VERSION_OVERRIDE is not None and (not re.fullmatch(r"[A-Za-z0-9_.-]+", DATA_VERSION_OVERRIDE)
                                          or DATA_VERSION_OVERRIDE in ("slim", ".", "..")):
    raise ValueError(f"DATA_VERSION_OVERRIDE invalide : {DATA_VERSION_OVERRIDE!r} (lettres, chiffres, _ - . seulement)")

PATHS = get_paths()
DATA_VERSION = DATA_VERSION_OVERRIDE or get_data_version()
OUT_DIR = PATHS["processed"] / DATA_VERSION
FIG_DIR = PATHS["results"] / "NB01" / DATA_VERSION
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"MODE         : {'DONNÉES RÉELLES' if DATA_VERSION != 'toy' else 'TOY (fausses données)'}")
print(f"DATA_VERSION : {DATA_VERSION}")
print(f"Dossier      : {OUT_DIR}")
print(f"Seed         : {SEED}")


In [ ]:
# ==============================================================================
# Cellule 2 : Chargement des sorties de prepare_vhh.py
# ==============================================================================
def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


meta_path, manifest_path = OUT_DIR / "vhh_clean_meta.csv", OUT_DIR / "manifest.json"
report_path, hist_path = OUT_DIR / "cleaning_report.csv", OUT_DIR / "length_histogram_before_length_filter.csv"
for p in (meta_path, manifest_path, report_path, hist_path):
    if not p.exists():
        raise FileNotFoundError(f"{p} introuvable : lancez d'abord prepare_vhh.py (NB01b) avec la même DATA_VERSION = {DATA_VERSION}.")

with open(manifest_path, encoding="utf-8") as f:
    manifest = json.load(f)
assert manifest["data_version"] == DATA_VERSION, "le manifest ne correspond pas à DATA_VERSION"
MIN_LEN, MAX_LEN = manifest["parameters"]["min_len"], manifest["parameters"]["max_len"]

clean = pd.read_csv(meta_path, dtype=str, keep_default_na=False)
clean["seq"] = clean["sequence"]
for col in ("length", "n_cys", "n_occurrences"):
    clean[col] = clean[col].astype(int)
clean["len"] = clean["length"]

print(f"{len(clean):,} séquences uniques | longueurs [{clean['len'].min()}, {clean['len'].max()}] | bornes : [{MIN_LEN}, {MAX_LEN}]")
print(f"Produit le {manifest['created_utc']} (commit {manifest['git_commit']}, script {manifest['script']})")
print("Fichiers sources :", clean["source_file"].value_counts().to_dict())

report = pd.read_csv(report_path)
report["% retiré"] = (100 * report["removed"] / report["rows_before"].clip(lower=1)).round(2)
display(report)
for n in manifest["notes"]:
    print("NOTE:", n)


In [ ]:
# ==============================================================================
# Cellule 3 : Distribution des longueurs AVANT le filtre de longueur (Tâche 1.3)
# ==============================================================================
h = pd.read_csv(hist_path)
lengths_all = np.repeat(h["length"].to_numpy(), h["n_sequences"].to_numpy())
mode_len_all = int(h.loc[h["n_sequences"].idxmax(), "length"])
n_short, n_long = int(h.loc[h["length"] < MIN_LEN, "n_sequences"].sum()), int(h.loc[h["length"] > MAX_LEN, "n_sequences"].sum())

display(pd.Series(lengths_all).describe(percentiles=[.25, .5, .75]).to_frame("longueur").T.round(2))
print(f"Longueur modale : {mode_len_all}")
print(f"Bornes (manifest) : MIN_LEN={MIN_LEN}, MAX_LEN={MAX_LEN}")
print(f"  trop courtes (< {MIN_LEN}) : {n_short:,}    trop longues (> {MAX_LEN}) : {n_long:,}")
display(clean.groupby("source_file")["len"].describe()[["count", "min", "50%", "max"]])

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(h["length"], h["n_sequences"], width=1.0, color="steelblue")
ax.axvline(MIN_LEN, color="red", ls="--", label=f"MIN_LEN = {MIN_LEN}")
ax.axvline(MAX_LEN, color="orange", ls="--", label=f"MAX_LEN = {MAX_LEN}")
ax.set_xlabel("Longueur (résidus)"); ax.set_ylabel("Nombre de séquences")
ax.set_title("Distribution des longueurs (après L1–L3, avant filtre de longueur)"); ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "lengths_histogram.png", dpi=150); plt.show()


**Décision sur les bornes.** `MIN_LEN` / `MAX_LEN` (fichier `utils.py`) sont *provisoires*. Si le pic principal est coupé ou si une sous-population importante est exclue, changez-les dans `utils.py` (une seule fois), relancez `prepare_vhh.py` dans une **nouvelle version** (ex. `v3`, ne jamais écraser `v2`) puis ce notebook, et notez la raison dans le rapport.

In [ ]:
# ==============================================================================
# Cellule 4 : Redondance exacte (Tâche 1.5) — la déduplication est faite par prepare_vhh.py
# ==============================================================================
n_before_dedup = int(manifest["parameters"].get("n_rows_before_dedup", clean["n_occurrences"].sum()))
n_unique = len(clean)
redundancy = (n_before_dedup - n_unique) / n_before_dedup
print(f"Séquences uniques : {n_unique:,} / {n_before_dedup:,}  ->  taux de redondance exacte = {redundancy:.2%}")
print(f"Séquences présentes plusieurs fois : {(clean['n_occurrences'] > 1).sum():,} ; occurrences max : {clean['n_occurrences'].max():,}")
display(clean["n_occurrences"].describe().to_frame("n_occurrences").T.round(2))
print("Séquences gardées par fichier :")
display(clean["source_file"].value_counts().to_frame("séquences"))


### Pourquoi la déduplication exacte ne suffit pas (à reformuler avec vos mots pour le rapport)
Elle n'élimine que les séquences identiques à 100 %. Dans un répertoire d'anticorps, l'hypermutation somatique produit des **quasi-doublons** (un ou deux résidus de différence) qui restent tous les deux. Avec un split aléatoire, un quasi-doublon peut se trouver dans *train* et l'autre dans *test* : le modèle a déjà vu presque la séquence de test, donc la vraisemblance test est trop optimiste (fuite d'information). La solution prévue est un clustering à 90 % d'identité (CD-HIT, Fu et al. 2012, ou MMseqs2) puis un split par clusters.

In [ ]:
# ==============================================================================
# Cellule 5 : Composition globale en acides aminés (Tâche 1.3)
# ==============================================================================
def composition(seqs: pd.Series) -> np.ndarray:
    """f(a) = count(a) / N_total, dans l'ordre de STANDARD_AA."""
    counts = np.array([int(seqs.str.count(a).sum()) for a in STANDARD_AA], dtype=np.int64)
    return counts / counts.sum()


p_bg = composition(clean["seq"])
display(pd.DataFrame({"f(a)": p_bg}, index=list(STANDARD_AA)).T.round(4))
print(f"Somme des fréquences : {p_bg.sum():.12f}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(list(STANDARD_AA), p_bg, color="steelblue")
ax.set_ylabel("Fréquence"); ax.set_title("Composition globale en acides aminés")
fig.tight_layout(); fig.savefig(FIG_DIR / "composition.png", dpi=150); plt.show()


### PWM et information par position (Tâche 1.4) — méthode et limite
Une PWM suppose des séquences **alignées**. Les gaps ayant été retirés, on utilise la **fenêtre alignable la plus simple** : toutes les séquences de la longueur modale L\*, comparées position par position. C'est une approximation : deux séquences de même longueur ne sont pas forcément alignées si leurs CDR diffèrent, donc la conservation des cadres (FR) est lisible mais celle des CDR est floue. Un vrai alignement IMGT (Phase 2) corrigera cela. Avec peu de séquences, H(i) est sous-estimée (donc I(i) surestimée).

In [ ]:
# ==============================================================================
# Cellule 6 : PWM, entropie de Shannon et information par position (Tâche 1.4)
# ==============================================================================
L_STAR = int(clean["len"].mode().iloc[0])
window = clean.loc[clean["len"] == L_STAR, "seq"]
M = len(window)
print(f"Fenêtre alignable : longueur modale L* = {L_STAR}, {M:,} séquences")
if M < 30:
    print("ATTENTION : moins de 30 séquences dans la fenêtre : PWM peu fiable (normal sur les données toy).")

arr = np.frombuffer("".join(window).encode("ascii"), dtype=np.uint8).reshape(M, L_STAR)
counts = np.stack([(arr == ord(a)).sum(axis=0) for a in STANDARD_AA], axis=1)      # (L*, 20)
p = counts / M

p_safe = np.where(p > 0, p, 1.0)                           # masque exact (pas d'epsilon) : 0·log 0 = 0
pwm = np.where(p > 0, np.log2(p_safe / p_bg[None, :]), -np.inf)
H = -np.sum(np.where(p > 0, p * np.log2(p_safe), 0.0), axis=1)
I = np.log2(20) - H

pwm_masked = np.ma.masked_invalid(np.where(np.isfinite(pwm), pwm, np.nan))
cmap = plt.get_cmap("RdBu_r").copy(); cmap.set_bad("lightgrey")
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(pwm_masked.T, aspect="auto", cmap=cmap, vmin=-3, vmax=3)
ax.set_yticks(range(20)); ax.set_yticklabels(list(STANDARD_AA))
ax.set_xlabel("Position (0 = premier résidu)"); ax.set_title("PWM : log2( p_i(a) / p_bg(a) ) — gris : p_i(a) = 0")
fig.colorbar(im, ax=ax); fig.tight_layout(); fig.savefig(FIG_DIR / "pwm_heatmap.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(np.arange(1, L_STAR + 1), I, color="steelblue")
ax.set_ylim(0, np.log2(20)); ax.set_xlabel("Position (1 = premier résidu)"); ax.set_ylabel("Information (bits)")
ax.set_title(f"Information par position I(i) = log2(20) − H(i)   (L* = {L_STAR}, {M:,} séquences)")
fig.tight_layout(); fig.savefig(FIG_DIR / "information_curve.png", dpi=150); plt.show()


In [ ]:
# ==============================================================================
# Cellule 7 : Validation automatique + nb01_stats.json (le manifest de prepare_vhh n'est PAS modifié)
# ==============================================================================
ALPHABET_PATTERN = "[" + STANDARD_AA + "]+"
sample = clean["seq"].iloc[:20000]
ref = compute_aa_composition(sample.tolist())
reread = pd.read_csv(OUT_DIR / "vhh_clean.csv", dtype=str, keep_default_na=False)

checks = {
    "100 % des séquences passent la regex des 20 AA": bool(clean["seq"].str.fullmatch(ALPHABET_PATTERN).all()),
    "toutes les longueurs sont dans les bornes": bool(clean["len"].between(MIN_LEN, MAX_LEN).all()),
    "aucun doublon exact": bool(clean["seq"].is_unique),
    "fréquences f(a) : somme = 1": bool(np.isclose(p_bg.sum(), 1.0)),
    "PWM : chaque position somme à 1": bool(np.allclose(p.sum(axis=1), 1.0)),
    "composition = utils.compute_aa_composition (échantillon)":
        bool(np.allclose([ref[a] for a in STANDARD_AA], composition(sample))),
    "vhh_clean.csv = séquences de vhh_clean_meta.csv": reread["sequence"].tolist() == clean["seq"].tolist(),
    "sha256 de vhh_clean.csv = manifest": sha256_of(OUT_DIR / "vhh_clean.csv") == manifest["outputs"]["vhh_clean.csv"]["sha256"],
}
for name, ok_ in checks.items():
    print(("OK   " if ok_ else "ÉCHEC"), name)
assert all(checks.values()), "Au moins une validation a échoué"
stats = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook": "NB01", "data_version": DATA_VERSION, "seed": SEED,
    "prepare_manifest_created_utc": manifest["created_utc"], "prepare_git_commit": manifest["git_commit"],
    "n_unique": int(n_unique), "n_before_dedup": n_before_dedup, "redundancy": float(redundancy),
    "pwm_window_length": L_STAR, "pwm_window_sequences": int(M), "checks": checks,
}
(OUT_DIR / "nb01_stats.json").write_text(json.dumps(stats, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nÉcrit : {OUT_DIR / 'nb01_stats.json'}")


In [ ]:
# ==============================================================================
# Cellule 8 : Fiche de synthèse — les chiffres à recopier dans le rapport (Tâche 1.6)
# ==============================================================================
order = np.argsort(p_bg)[::-1]
top_pos = np.argsort(I)[::-1][:5]
low_pos = np.argsort(I)[:5]
raw_rows = int(report.loc[0, "rows_before"])
fiche = {
    "version des données": DATA_VERSION,
    "lignes brutes": f"{raw_rows:,}",
    "séquences après L1–L6 (avant dédup)": f"{n_before_dedup:,}",
    "séquences uniques": f"{n_unique:,}",
    "rejet L1–L6": f"{1 - n_before_dedup / raw_rows:.2%}",
    "longueur min / médiane / modale / max": f"{clean['len'].min()} / {int(clean['len'].median())} / {L_STAR} / {clean['len'].max()}",
    "redondance exacte": f"{redundancy:.2%}",
    "3 AA les plus fréquents": ", ".join(f"{STANDARD_AA[i]} ({p_bg[i]:.1%})" for i in order[:3]),
    "3 AA les plus rares": ", ".join(f"{STANDARD_AA[i]} ({p_bg[i]:.1%})" for i in order[-3:]),
    "5 positions les plus informatives (1-indexées)": ", ".join(f"{i + 1} ({I[i]:.2f} bits)" for i in top_pos),
    "5 positions les moins informatives": ", ".join(f"{i + 1} ({I[i]:.2f} bits)" for i in low_pos),
}
for k, v in fiche.items():
    print(f"{k:<48} {v}")


## Analyse rédigée — Tâche 1.6 (demi-page du rapport)
*À rédiger vous-mêmes à partir de la fiche de synthèse ci-dessus (données réelles, pas toy).*

1. **Longueurs** : la distribution est-elle unimodale ? Y a-t-il des sous-populations ? Les bornes choisies sont-elles justifiées par l'histogramme ?
2. **Conservation** : quelles positions ont une information élevée ? Retrouve-t-on des cadres conservés (FR) et des zones variables (CDR) ? Quelles sont les limites de la fenêtre de longueur fixe ?
3. **Redondance** : quel est le taux de doublons exacts ? Pourquoi la déduplication à identité partielle sera-t-elle nécessaire ?

> Rappel : ne jamais recopier ici les chiffres d'un run **toy**.